In [5]:
import torch 
import torchvision
import torch.nn as nn
import numpy as np
import math
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [6]:
class configConvNet(nn.Module):
    residual = False;
    def __init__(self, in_channels=3, base_channels=16, kernel_size=2, batch_normal=True, num_classes=18, num_layers=3, activation="RELU", residuals=False):
        super(configConvNet, self).__init__()
        input_channels = in_channels;
        layer = []
        self.residual = residuals;
        out_channels = 128;
        for i in range(num_layers):
            out_channels = base_channels * (2 ** i)
            layer.append(nn.Conv2d(input_channels, out_channels, kernel_size=5, stride=1, padding=2))
            if batch_normal:
                layer.append(nn.BatchNorm2d(out_channels))
            if activation.upper() == "RELU":
                layer.append(nn.ReLU())
            elif activation.upper() == "LEAKYRELU":
                layer.append(nn.LeakyReLU(0.1))
            elif activation.upper() == "GELU":
                layer.append(nn.GELU())
            layer.append(nn.MaxPool2d(kernel_size=2, stride=2))
            input_channels = out_channels
        if self.residual:
            self.downsample = None
            if in_channels != out_channels:
                print(f"in_channels:{in_channels}, out_channels:{out_channels}")
                self.downsample = nn.Sequential(
                    nn.Conv2d(
                        in_channels, 
                        out_channels, 
                        kernel_size=1, 
                        padding='same', 
                        bias=False
                    ),
                    nn.BatchNorm2d(out_channels) if batch_normal else nn.Identity(),
                    nn.MaxPool2d(kernel_size=2, stride=(2 ** num_layers)),
                )
            
        self.features = nn.Sequential(*layer)
        sqrt = 128 // (2 ** num_layers) #this isn't what a square root is.
        print(sqrt)
        self.fc = nn.Linear(out_channels*sqrt*sqrt, num_classes)

    def forward(self, x):
        identity = x
        out = self.features(x)
        
        if self.residual:
            identity = self.downsample(identity) if self.downsample else identity
            #print("identity shape:", identity.shape, "out shape", out.shape)
            out += identity 
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

In [7]:
import os
import torch
import pandas as pd
from skimage import io, transform
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision import transforms, utils, datasets
from collections import Counter
import re

class ImageFolderExclude(datasets.ImageFolder):
    def __init__(self, root, exclude_pattern=None, **kwargs):
        self.exclude_pattern = re.compile(exclude_pattern) if exclude_pattern else None
        super().__init__(root, **kwargs)

    def find_classes(self, directory):
        classes = [entry.name for entry in os.scandir(directory) if entry.is_dir()]
        if self.exclude_pattern:
            classes = [cls for cls in classes if self.exclude_pattern.search(cls)]
        classes.sort()
        class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
        return classes, class_to_idx

class KaakaaDataset(Dataset):
    def __init__(self, root_dir, csv_file=None, prune=None, transform=None, target_transform=None):
        self.class_counts = Counter()
        self.root_dir = root_dir
        self.csv_file = csv_file #if I ever use the Cornell dataset...
        self.transform = transform
        self.target_transform = target_transform
        
        if prune:
            self.dataset = ImageFolderExclude(root=self.root_dir, exclude_pattern=prune, transform=self.transform)
        else: 
            self.dataset = datasets.ImageFolder(root=self.root_dir, transform=self.transform)
        
        all_datasets = self.dataset.datasets if isinstance(self.dataset, ConcatDataset) else [self.dataset]
        #counting up what we have for each class
        for ds in all_datasets:
            for _, label in ds.samples:
                self.class_counts[ds.classes[label]] += 1
    
    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        #don't need to apply transforms as ImageFolder already does that
        return image, torch.tensor(label)

In [8]:
from torch.utils.data import random_split, DataLoader

new_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

train_ratio = 0.6
val_ratio = 0.2
test_ratio = 0.2
random_seed = 42

root_dir = "E:/Datasets/Dataset_B/masked-split_train_val_by_day"

train_dir = os.path.join(root_dir, "train")
val_dir = os.path.join(root_dir, "val")
test_dir = os.path.join(root_dir, "test")

# Create dataset instances
train_dataset = KaakaaDataset(root_dir=train_dir, transform=new_transforms)
val_dataset = KaakaaDataset(root_dir=val_dir, transform=new_transforms)
test_dataset = KaakaaDataset(root_dir=test_dir, transform=new_transforms)


#Setting up for our weightedrandomsampler, which will help us with class imbalance
labels = [label for _, label in train_dataset]

class_counts = torch.bincount(torch.tensor(labels))
num_classes = len(class_counts)

class_weights = 1 / class_counts
print(class_weights)

#weights are inverse of class frequency
class_weights = 1.0 / class_counts.float()
#assign a weight to each sample based on its label
sample_weights = class_weights[torch.tensor(labels)]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(class_weights),
    replacement=True  # allows oversampling
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True) #we're using a sampler, so no shuffle
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


tensor([0.0714, 0.0018, 0.0050, 0.2500, 0.0025, 0.0090, 0.0065, 0.1250, 0.0052,
        0.0012, 0.0294, 0.0028, 0.1111, 0.0101, 0.0161, 0.0014, 0.0044, 0.0109])


In [9]:
def append_text_to_file(file_path, text_to_append):
    try:
        with open(file_path, 'a') as file:
            file.write(text_to_append + '\n')
        print(f"Text appended to {file_path} successfully.")
    except Exception as e:
        print(f"Error: {e}")

In [10]:
# Plot training loss
def generate_charts(num_layers, activation, batch_norm, residuals, train_losses, val_accuracies):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label=f'Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Training Loss Curve \n({num_layers} layers, {activation}, batchnorm={batch_norm}, residuals={residuals})')
    plt.legend()
    
    # Plot validation accuracy
    plt.subplot(1, 2, 2)
    plt.plot(val_accuracies, label=f'Validation Accuracy', color='green')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title(f'Validation Accuracy Curve \n({num_layers} layers, {activation}, batchnorm={batch_norm}, residuals={residuals})')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(f'layers{num_layers}_batch{batch_norm}_residuals{residuals}_{activation}.png')
    plt.show()

In [11]:
import re
import os
#i used this to 'save' my progress - in case something interrupted training
#so i wouldn't have to start at model 1 out of 48 or figure out where i left off each time
def extract_models_from_file(filepath):
    models_ive_trained = []

    #creates file if it doesn't exist
    if not os.path.exists(filepath):
        with open(filepath, "w") as file:
            pass  

    # Read file
    with open(filepath, "r") as file:
        lines = file.readlines()

    # Define a regex pattern to capture required parameters
    pattern = re.compile(
        r"Batch normalisation: (\w+); residuals: (\w+); num_layers:(\d+); activation: (\w+)",
        re.IGNORECASE
    )

    # Process every line to match pattern
    for line in lines:
        match = pattern.search(line)
        if match:
            batch_norm, residuals, num_layers, activation = match.groups()
            model_string = f"{batch_norm}{residuals}{num_layers}{activation}"
            models_ive_trained.append(model_string)

    return models_ive_trained

In [12]:
def test_model(model, descript, batch_size, device):
    
    save_dir = "saved_models"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, descript)
    torch.save(model, save_path)
    
    model.eval()

    # Disable gradient calculation for efficiency
    with torch.no_grad():
        correct = 0
        total = 0
        
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
    
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            
        new_acc_descript = '       Test Accuracy of the model on the 10000 test images: {:.2f} %'.format(100 * correct / total)
        append_text_to_file('AllAccuracies.txt', descript + "\n" + new_acc_descript)
        print(descript + "\n" + new_acc_descript)

In [ ]:
def run_model():
    #hyperparameters
    num_epochs = 10
    num_classes = 18
    learning_rate = 0.0001
    batch_size = 8

    #this is stupid as hell but my computer died trying to do 7 layers last night and i want to do every permutation
    #which means. this must be automated
    #so now we're going to check what we tested last time we rendered
    models_ive_trained = extract_models_from_file("AllAccuracies.txt")

    bools = [ False, True ]
    num_layers = [3, 4, 5, 6 ] #removed 7 as returns were diminishing on 6 and 7 gets an out of memory error
    activation_funcs = ["ReLU", "LeakyReLU", "GELU"]

    #selecting device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    #printing because my main kernel wants to be stuck on CPU-only pytorch fsr
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
    for z in range(len(bools)):
        batch_bool = bools[z]
        for m in range(len(bools)):
            residual = bools[m]
            for j in range(len(num_layers)):
                num_layer = num_layers[j]
                for k in range(len(activation_funcs)):
                    activation_func = activation_funcs[k]
                    this_model = f"{batch_bool}{residual}{num_layer}{activation_func}"
                    if this_model in models_ive_trained:
                        print("We've already trained this model, it's getting skipped.")
                    else:
                        model = configConvNet(batch_normal=batch_bool, num_layers=num_layer, activation=activation_func, residuals=residual).to(device)
        
                        # Loss and optimizer
                        criterion = nn.CrossEntropyLoss()
                        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
                        train_losses = []
                        val_accuracies = []
                        
                        # Train the model
                        total_step = len(train_loader)
                        descript = f"Batch normalisation: {bools[z]}; residuals: {bools[m]}; num_layers:{num_layers[j]}; activation: {activation_funcs[k]}";
                        print(descript)
                        
                        for epoch in range(num_epochs):
                            model.train()
                            epoch_loss = 0
                            for i, (images, labels) in enumerate(train_loader):
                                print(i)
                                images = images.to(device)
                                labels = labels.to(device)
                                #print("images ", images.shape)
                                # Forward pass
                                outputs = model(images)
                                #print("outputs ", outputs.shape)
                                loss = criterion(outputs, labels)
        
                                # Backward and optimize
                                optimizer.zero_grad()
                                loss.backward()
                                optimizer.step()
                                epoch_loss += loss.item()                            
                                if (i+1) % 86 == 0:
                                    print(torch.softmax(outputs, dim=1)[0])
                                    print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'.format(epoch+1, num_epochs, i+1, total_step, loss.item()))
                            
                            avg_loss = epoch_loss / total_step
                            train_losses.append(avg_loss)
                                
                            #now we do validation. exit training mode
                            model.eval()
                            correct = 0                       
                            total = 0
                            with torch.no_grad():
                                for images, labels in val_loader:
                                    images = images.to(device)
                                    labels = labels.to(device)
                                    outputs = model(images)
                                    _, predicted = torch.max(outputs.data, 1)
                                    total += labels.size(0)
                                    correct += (predicted == labels).sum().item()
                            accuracy = 100 * correct / total
                            val_accuracies.append(accuracy)
                            #print(f'Validation Accuracy after epoch {epoch+1}: {accuracy:.2f}%')
                        generate_charts(num_layers=num_layer, activation=activation_func,batch_norm=batch_bool, residuals=residual, train_losses=train_losses, val_accuracies=val_accuracies)                    
                        test_model(model, descript, batch_size, device)
run_model()

GPU name: NVIDIA GeForce RTX 3070 Laptop GPU
16
Batch normalisation: False; residuals: False; num_layers:3; activation: ReLU
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
tensor([0.0065, 0.1884, 0.0910, 0.0007, 0.0403, 0.0195, 0.0632, 0.0002, 0.0259,
        0.2176, 0.0098, 0.0563, 0.0014, 0.0231, 0.0118, 0.1477, 0.0736, 0.0228],
       device='cuda:0', grad_fn=<SelectBackward0>)
Epoch [1/10], Step [86/258], Loss: 1.8115
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
tensor([5.7533e-04, 8.1689e-01, 7.5242e-03,